# Inter-Annotator Agreement (IAA) for Extracted Claim Terms

Compares the concept terms extracted by three annotators (**Tianyu**, **Mark**, **Gloria**)
from 100 patent claims.

1. Normalize each annotator's raw text cell into a clean list of terms (including removing
   parenthesized abbreviations such as `(PS)`).
2. Find exact term matches per claim — the primary, trustworthy metric (zero ambiguity).
   Differences in case, hyphen vs space, and trailing punctuation are ignored.
3. Compute strict pairwise Precision / Recall / F1 from those exact matches.
4. Summarize the strict scores — the headline number.
5. Define the match criteria of Tsai et al. (2006) by position in the claim text: exact,
   right, left, left/right, approximate, core-term and partial match.
6. Locate each term in the claim text and count matches under each criterion.
7. Score each criterion independently with pooled (corpus-level) Precision / Recall / F1,
   as in the paper.

In [1]:
import re
from pathlib import Path
from itertools import combinations # For unique pairs of annotators

import numpy as np
import pandas as pd
from scipy.optimize import linear_sum_assignment

pd.set_option("display.max_colwidth", 200)

# The CSV sits in the repository root, one level above this notebook.
RAW_PATH = Path("..") / "human_claim_annotations_100 - Tabellenblatt1.csv"
ANNOTATORS = ["Tianyu", "Mark", "Gloria"]
PAIRS = list(combinations(ANNOTATORS, 2))  # (Tianyu, Mark), (Tianyu, Gloria), (Mark, Gloria)

df_raw = pd.read_csv(RAW_PATH)
df_raw.head()


,patent_id,claim_number,claim_text,Tianyu,Mark,Gloria
0,7696175,33,"33. The method of claim 32 , wherein said soluble form of the co-stimulatory molecule is linked to another protein.",soluble form; co-stimulatory molecule; protein,co-stimulatory molecule; protein; soluble form,co-stimulatory molecule
1,7704241,5,"5. The absorbent article according to claim 4 , wherein said absorbent system comprises a material including a mixture of cellulosic fibers and superabsorbent material.",absorbent article; absorbent system; cellulosic fibers; superabsorbent material,absorbent system; cellulosic fibers; superabsorbent material,absorbent system; cellulosic fibers; superabsorbent material
2,7767672,54,"54. The compound of claim 52 , wherein said halogen of J is F.",halogen,halogen,"halogen F, halogen J"
3,7782077,31,"31. The method of claim 28 , further comprising filtering the first input pulse through a first input filter.",input pulse,input pulse; input filter,input pulse; input filter; filtering
4,7866377,9,"9. The method of claim 8 wherein the three-dimensional minimal skeleton heat exchanger forms a plate heat exchanger composed of multiple, thin slightly-separated plates that have large surface are...",heat exchanger; fluid flow passages; heat transfer,heat exchanger; flow passages; heat transfer; three-dimensional minimal skeleton; three-dimensional minimal skeleton heat exchanger,plate heat exchanger


## Step 1 - Normalize separators and abbreviations

Split on `;` always, and on `,` only when it's immediately followed by whitespace - a bare
`,` with no following space is part of a chemical name (e.g. `2,4-dichlorophenyl`,
`3,5-bis(trifluoromethyl)phenyl`) and must be left untouched. Each resulting term is
stripped and the list is de-duplicated case-insensitively, preserving order.

**Abbreviations.** A parenthesized abbreviation after a term is removed, so
`polymer stabilized (PS) type` becomes `polymer stabilized type` and can match exactly.
Parenthesized text counts as an abbreviation only when:

- it follows a space,
- it is one word of at least two characters with an uppercase letter (`PS`, `DLBCL`, `scFv`),
- it is not a Roman numeral (so `compound of formula (I)` / `(II)` stay distinct).

In [2]:
# Split on ';' always; split on ',' only when followed by whitespace (a tight comma,
# e.g. "2,4-dichlorophenyl", is part of a chemical name and must not be split on).
SPLIT_RE = re.compile(r";\s*|,(?=\s)")

# " (XYZ)": a single parenthesized word preceded by whitespace.
PARENTHESIZED_WORD_RE = re.compile(r"\s+\(([^()\s]+)\)")
ROMAN_NUMERAL_RE = re.compile(r"[IVX]+")


def is_abbreviation(text):
    """True for text like 'PS', 'DLBCL' or 'scFv'; False for 'I', 'II' or lowercase words."""
    has_uppercase = any(ch.isupper() for ch in text)
    return len(text) >= 2 and has_uppercase and not ROMAN_NUMERAL_RE.fullmatch(text)


def remove_abbreviations(term):
    """Drop parenthesized abbreviations: 'polymer stabilized (PS) type' -> 'polymer stabilized type'."""
    def replace(match):
        return "" if is_abbreviation(match.group(1)) else match.group(0)
    return PARENTHESIZED_WORD_RE.sub(replace, term).strip()


def normalize_terms(cell):
    """Split an annotator cell into a de-duplicated, order-preserving list of terms."""
    if not isinstance(cell, str) or not cell.strip():
        return []
    raw_terms = [remove_abbreviations(t.strip()) for t in SPLIT_RE.split(cell)]
    raw_terms = [t for t in raw_terms if t]
    seen, deduped = set(), []
    for term in raw_terms:
        key = term.lower()
        if key not in seen:
            seen.add(key)
            deduped.append(term)
    return deduped


# Sanity check: a purely-numeric "term" means the split rule broke a chemical name.
numeric_flags = []
for idx, row in df_raw.iterrows():
    for col in ANNOTATORS:
        for term in normalize_terms(row[col]):
            if term.replace(".", "", 1).isdigit():
                numeric_flags.append({
                    "row": idx, "patent_id": row["patent_id"], "annotator": col,
                    "flagged_term": term, "raw_cell": row[col],
                })

print(f"Numeric-term flags: {len(numeric_flags)} (0 expected on the current dataset)")
for flag in numeric_flags:
    print(flag)

print(remove_abbreviations("polymer stabilized (PS) type"))
print(remove_abbreviations("poly(ethylene glycol) (PEG)"))
print(remove_abbreviations("compound of formula (I)"))

Numeric-term flags: 0 (0 expected on the current dataset)
polymer stabilized type
poly(ethylene glycol)
compound of formula (I)


In [3]:
# Sanity check for chemical terms - is the above parsing path working? 
tight_comma_mask = df_raw[ANNOTATORS].apply(
    lambda col: col.str.contains(r",(?!\s)", regex=True, na=False)
).any(axis=1)

df_raw.loc[tight_comma_mask, ["patent_id", "claim_number"] + ANNOTATORS]


,patent_id,claim_number,Tianyu,Mark,Gloria
6,7915258,8,1-propylbutyl; cyclohexyl; 4-tert-butylcyclohexyl; 4-(trifluoromethyl)cyclohexyl; adamantan-1-yl; phenyl; 4-fluorophenyl; 2-methylphenyl; 4-methylphenyl; 4-isopropylphenyl; 4-butylphenyl; 4-tert-b...,"1-propylbutyl; cyclohexyl, 4-tert-butylcyclohexyl, 4-(trifluoromethyl)cyclohexyl; adamantan-1-yl; phenyl, 4-fluorophenyl, 2-methylphenyl, 4-methylphenyl, 4 isopropylphenyl, 4-butylphenyl, 4-tert-b...",compound of formula (I); 1-propylbutyl; cyclohexyl; 4-tert-butylcyclohexyl; 4-(trifluoromethyl)cyclohexyl; adamantan-1-yl; phenyl; 4-fluorophenyl; 2-methylphenyl; 4-methylphenyl; 4-isopropylphenyl...
36,8871972,10,"adapalene; adapalene methyl ester; hydrolyzing; adapalene salt; 3,3′-diadamantyl-4,4′-dimethoxybiphenyl; 3,3′-diadamantyl-4,4′-dimethoxybiphenyl impurity","method; adapalene; pharmaceutical use; adapalene methyl ester; hydrolyzing; adapalene salt; adapalene; 3,3′-diadamantyl-4,4′-dimethoxybiphenyl; reference marker; 3,3′-diadamantyl-4,4′-dimethoxybip...","adapalene; adapalene methyl ester; adapalene salt; 3,3′-diadamantyl-4,4′-dimethoxybiphenyl; reference marker; impurity"
65,9561309,20,"biocompatible polymer; poly(ester amides); polystyrene- polyisobutylene-polystyrene; block copolymers; polystyrene; polyisobutylene; polycaprolactone; poly(L-lactide); poly(D,L-lactide); poly(lact...",polymeric composition; biocompatible polymer; poly(ester amides); polystyrene- polyisobutylene-polystyrene; block copolymers (SIS); polystyrene; polyisobutylene; polycaprolactone (PCL); poly(L-lac...,polymeric composition; biocompatible polymer; poly(ester amides); polystyrene-polyisobutylene-polystyrene; block copolymers; polycaprolactone (PCL); polylactic acid (PLA); poly(glycolide); polydim...
87,10197567,9,"screening method; azoline compound; azoline compound library; azoline backbone; Cys; Ser; Thr; 2,3-diamino acid; Xaa 0; peptide; -(Xaa 0 ) m -; Xaa 0; amino acid; azoline ring; heterocyclase; hydr...","screening method; azoline compound; target substance; azoline compound library; azoline backbone; Cys; Ser; Thr; 2,3-diamino acid; analogs; Xaa 0; peptide; -(Xaa 0 ) m -; arbitrary amino acid; azo...",screening method; azoline compound; target substance; azoline compound library; azoline backbone; peptide; amino acid; azoline ring; heterocyclase; mRNA library; precursor peptides; cell-free tran...
97,10477209,3,image filtering; deblocked decoded image; pixel value; plurality of unit areas; image filtering device; input image; pixel value; subject pixel; classifying; offset class; subject pixel; offset va...,image filtering method; deblocked decoded image; pixel value; unit area; deblocked decoded image; image filtering device; offset value range; offset bit depth−K−1) −1); offset bit depth;\n SAO_DE...,image filtering method; deblocked decoded image; offset value; subject pixel; offset classes; offset bit depth


In [4]:
df_clean = df_raw.copy()
for col in ANNOTATORS:
    df_clean[col] = df_raw[col].apply(lambda cell: "; ".join(normalize_terms(cell)))

df_clean.head()


,patent_id,claim_number,claim_text,Tianyu,Mark,Gloria
0,7696175,33,"33. The method of claim 32 , wherein said soluble form of the co-stimulatory molecule is linked to another protein.",soluble form; co-stimulatory molecule; protein,co-stimulatory molecule; protein; soluble form,co-stimulatory molecule
1,7704241,5,"5. The absorbent article according to claim 4 , wherein said absorbent system comprises a material including a mixture of cellulosic fibers and superabsorbent material.",absorbent article; absorbent system; cellulosic fibers; superabsorbent material,absorbent system; cellulosic fibers; superabsorbent material,absorbent system; cellulosic fibers; superabsorbent material
2,7767672,54,"54. The compound of claim 52 , wherein said halogen of J is F.",halogen,halogen,halogen F; halogen J
3,7782077,31,"31. The method of claim 28 , further comprising filtering the first input pulse through a first input filter.",input pulse,input pulse; input filter,input pulse; input filter; filtering
4,7866377,9,"9. The method of claim 8 wherein the three-dimensional minimal skeleton heat exchanger forms a plate heat exchanger composed of multiple, thin slightly-separated plates that have large surface are...",heat exchanger; fluid flow passages; heat transfer,heat exchanger; flow passages; heat transfer; three-dimensional minimal skeleton; three-dimensional minimal skeleton heat exchanger,plate heat exchanger


## Step 2 — Exact matching (the primary, trustworthy metric)

Two terms match exactly when they are identical after ignoring differences that are only
formatting: case, hyphen vs space, repeated whitespace, and trailing punctuation. So
`4-isopropylphenyl` / `4 isopropylphenyl` and `cells.` / `cells` are exact matches.

In [5]:
HYPHEN_OR_SPACE_RE = re.compile(r"[\s\-]+")


def exact_key(term):
    """Comparison key that ignores formatting only: '4-Isopropylphenyl.' -> '4 isopropylphenyl'."""
    term = term.strip().lower().rstrip(".,;:").strip()
    return HYPHEN_OR_SPACE_RE.sub(" ", term)


def find_exact_matches(list_a, list_b):
    """  
    This function compares the terms from two annotators for 
    the same claim and identifies which terms match exactly. It ignores 
    formatting differences (see exact_key), ensures each term is matched only once, 
    and returns both the exact matches and the unmatched terms.
    """
    a_items, b_items = list(enumerate(list_a)), list(enumerate(list_b))
    matched_a, matched_b = set(), set()
    exact_matches = []

    for i, a_term in a_items:
        a_norm = exact_key(a_term)
        for j, b_term in b_items:
            if j in matched_b:
                continue
            if exact_key(b_term) == a_norm:
                exact_matches.append((a_term, b_term))
                matched_a.add(i)
                matched_b.add(j)
                break

    unmatched_a = [t for i, t in a_items if i not in matched_a]
    unmatched_b = [t for j, t in b_items if j not in matched_b]
    return exact_matches, unmatched_a, unmatched_b


# Quick example: claim 0 (Tianyu vs Mark) and the halogen claim (patent 7767672, claim 54)
demo_a = normalize_terms(df_raw.loc[0, "Tianyu"])
demo_b = normalize_terms(df_raw.loc[0, "Mark"])
print("Claim 0, Tianyu vs Mark:", find_exact_matches(demo_a, demo_b))

halogen_row = df_raw[df_raw["patent_id"] == 7767672].iloc[0]
h_tianyu = normalize_terms(halogen_row["Tianyu"])
h_gloria = normalize_terms(halogen_row["Gloria"])
print("Halogen claim, Tianyu vs Gloria:", find_exact_matches(h_tianyu, h_gloria))


Claim 0, Tianyu vs Mark: ([('soluble form', 'soluble form'), ('co-stimulatory molecule', 'co-stimulatory molecule'), ('protein', 'protein')], [], [])
Halogen claim, Tianyu vs Gloria: ([], ['halogen'], ['halogen F', 'halogen J'])


## Step 3 — Strict Precision / Recall / F1 (primary metric)

For each annotator pair (Tianyu-Mark, Tianyu-Gloria, Mark-Gloria) and each claim:

- `matches` = exact matches only (Step 2)
- `Precision = matches / len(terms_B)`, `Recall = matches / len(terms_A)`,
  `F1 = 2*P*R/(P+R)`

Since there's no ground truth, both **reference directions** are computed per pair and kept
in `df_detail` for drill-down. F1 is identical in both directions since it's symmetric.

If both annotators extracted nothing for a claim, that claim counts as full agreement
(P = R = F1 = 1).

This is the trustworthy, headline number — zero ambiguity, no partial-credit judgment call.
Steps 5-7 add relaxed scores using the match criteria of Tsai et al. (2006).

In [6]:
def safe_div(numerator, denominator):
    return numerator / denominator if denominator else 0.0


def f1_from_pr(precision, recall):
    return safe_div(2 * precision * recall, precision + recall)


def prf(n_matches, num_ref, num_other):
    """Precision, recall, F1 for one claim; two empty lists count as full agreement."""
    if num_ref == 0 and num_other == 0:
        return 1.0, 1.0, 1.0
    precision = safe_div(n_matches, num_other)
    recall = safe_div(n_matches, num_ref)
    return precision, recall, f1_from_pr(precision, recall)


detail_rows = []
for idx, row in df_raw.iterrows():
    terms = {ann: normalize_terms(row[ann]) for ann in ANNOTATORS}
    for ann_a, ann_b in PAIRS:
        list_a, list_b = terms[ann_a], terms[ann_b]
        exact, unmatched_a, unmatched_b = find_exact_matches(list_a, list_b)
        n_exact = len(exact)
        len_a, len_b = len(list_a), len(list_b)

        for ref, num_ref, num_other in [("a", len_a, len_b), ("b", len_b, len_a)]:
            precision_strict, recall_strict, f1_strict = prf(n_exact, num_ref, num_other)

            detail_rows.append({
                "patent_id": row["patent_id"],
                "claim_number": row["claim_number"],
                "annotator_a": ann_a,
                "annotator_b": ann_b,
                "reference": ann_a if ref == "a" else ann_b,
                "n_terms_a": len_a,
                "n_terms_b": len_b,
                "exact_matches": exact,
                "unmatched_to_a": unmatched_a,
                "unmatched_to_b": unmatched_b,
                "n_exact": n_exact,
                "precision_strict": precision_strict,
                "recall_strict": recall_strict,
                "f1_strict": f1_strict,
            })

df_detail = pd.DataFrame(detail_rows)
df_detail.head(10)


,patent_id,claim_number,annotator_a,annotator_b,reference,n_terms_a,n_terms_b,exact_matches,unmatched_to_a,unmatched_to_b,n_exact,precision_strict,recall_strict,f1_strict
0,7696175,33,Tianyu,Mark,Tianyu,3,3,"[(soluble form, soluble form), (co-stimulatory molecule, co-stimulatory molecule), (protein, protein)]",[],[],3,1.000000,1.000000,1.000000
1,7696175,33,Tianyu,Mark,Mark,3,3,"[(soluble form, soluble form), (co-stimulatory molecule, co-stimulatory molecule), (protein, protein)]",[],[],3,1.000000,1.000000,1.000000
2,7696175,33,Tianyu,Gloria,Tianyu,3,1,"[(co-stimulatory molecule, co-stimulatory molecule)]","[soluble form, protein]",[],1,1.000000,0.333333,0.500000
3,7696175,33,Tianyu,Gloria,Gloria,3,1,"[(co-stimulatory molecule, co-stimulatory molecule)]","[soluble form, protein]",[],1,0.333333,1.000000,0.500000
4,7696175,33,Mark,Gloria,Mark,3,1,"[(co-stimulatory molecule, co-stimulatory molecule)]","[protein, soluble form]",[],1,1.000000,0.333333,0.500000
5,7696175,33,Mark,Gloria,Gloria,3,1,"[(co-stimulatory molecule, co-stimulatory molecule)]","[protein, soluble form]",[],1,0.333333,1.000000,0.500000
6,7704241,5,Tianyu,Mark,Tianyu,4,3,"[(absorbent system, absorbent system), (cellulosic fibers, cellulosic fibers), (superabsorbent material, superabsorbent material)]",[absorbent article],[],3,1.000000,0.750000,0.857143
7,7704241,5,Tianyu,Mark,Mark,4,3,"[(absorbent system, absorbent system), (cellulosic fibers, cellulosic fibers), (superabsorbent material, superabsorbent material)]",[absorbent article],[],3,0.750000,1.000000,0.857143
8,7704241,5,Tianyu,Gloria,Tianyu,4,3,"[(absorbent system, absorbent system), (cellulosic fibers, cellulosic fibers), (superabsorbent material, superabsorbent material)]",[absorbent article],[],3,1.000000,0.750000,0.857143
9,7704241,5,Tianyu,Gloria,Gloria,4,3,"[(absorbent system, absorbent system), (cellulosic fibers, cellulosic fibers), (superabsorbent material, superabsorbent material)]",[absorbent article],[],3,0.750000,1.000000,0.857143


## Step 4 — Strict summary (primary, headline metric)

`df_summary_strict` reports precision, recall, and F1 from exact matches only, averaged
across all 100 claims, per annotator pair *and* reference direction, plus each annotator's
average across their two pairings. **This is the number to lead with** — zero ambiguity, no
partial-credit judgment call. Steps 5-7 build relaxed scores on top of it.

Averaging per claim gives a 1-term claim the same weight as a 30-term claim, so
`df_summary_pooled` also reports **pooled (micro-averaged)** scores: matches and term counts
are summed over all claims first, then P / R / F1 are computed once per pair. This is the
more standard figure for extraction agreement; report it alongside the per-claim average.

In [7]:
strict_cols = ["precision_strict", "recall_strict", "f1_strict"]

# Pair-level averages, kept per reference direction since precision/recall depend on
# which annotator's terms are treated as the reference (F1 is symmetric and repeats).
pair_summary_strict = df_detail.groupby(["annotator_a", "annotator_b", "reference"])[strict_cols].mean().reset_index()
pair_summary_strict.insert(0, "scope", "pair")
pair_summary_strict["n_claims"] = len(df_raw)

# Per-annotator averages across their two pairings, using that annotator as the reference.
annotator_rows = []
for annotator in ANNOTATORS:
    relevant = pair_summary_strict[pair_summary_strict["reference"] == annotator]
    means = relevant[strict_cols].mean()
    annotator_rows.append({
        "scope": "annotator",
        "annotator_a": annotator,
        "annotator_b": None,
        "reference": annotator,
        "n_claims": len(df_raw),
        **{col: means[col] for col in strict_cols},
    })

df_summary_strict = pd.concat([pair_summary_strict, pd.DataFrame(annotator_rows)], ignore_index=True)
display(df_summary_strict)


def pooled_scores(detail, n_col, suffix):
    """Micro-averaged P / R / F1 per pair and reference direction: sum counts, then divide."""
    rows = []
    for (ann_a, ann_b, ref), grp in detail.groupby(["annotator_a", "annotator_b", "reference"]):
        n_matches = grp[n_col].sum()
        n_ref = grp["n_terms_a"].sum() if ref == ann_a else grp["n_terms_b"].sum()
        n_other = grp["n_terms_b"].sum() if ref == ann_a else grp["n_terms_a"].sum()
        precision, recall = safe_div(n_matches, n_other), safe_div(n_matches, n_ref)
        rows.append({
            "annotator_a": ann_a, "annotator_b": ann_b, "reference": ref,
            f"precision_{suffix}": precision,
            f"recall_{suffix}": recall,
            f"f1_{suffix}": f1_from_pr(precision, recall),
        })
    return pd.DataFrame(rows)


df_summary_pooled = pooled_scores(df_detail, "n_exact", "strict_pooled")
df_summary_pooled.round(3)

,scope,annotator_a,annotator_b,reference,precision_strict,recall_strict,f1_strict,n_claims
0,pair,Mark,Gloria,Gloria,0.590383,0.752497,0.643928,100
1,pair,Mark,Gloria,Mark,0.752497,0.590383,0.643928,100
2,pair,Tianyu,Gloria,Gloria,0.674700,0.746103,0.686413,100
3,pair,Tianyu,Gloria,Tianyu,0.746103,0.674700,0.686413,100
4,pair,Tianyu,Mark,Mark,0.795157,0.693613,0.728149,100
5,pair,Tianyu,Mark,Tianyu,0.693613,0.795157,0.728149,100
6,annotator,Tianyu,None,Tianyu,0.719858,0.734928,0.707281,100
7,annotator,Mark,None,Mark,0.773827,0.641998,0.686039,100
8,annotator,Gloria,None,Gloria,0.632541,0.749300,0.665171,100


,annotator_a,annotator_b,reference,precision_strict_pooled,recall_strict_pooled,f1_strict_pooled
0,Mark,Gloria,Gloria,0.545,0.789,0.644
1,Mark,Gloria,Mark,0.789,0.545,0.644
2,Tianyu,Gloria,Gloria,0.583,0.781,0.668
3,Tianyu,Gloria,Tianyu,0.781,0.583,0.668
4,Tianyu,Mark,Mark,0.806,0.745,0.774
5,Tianyu,Mark,Tianyu,0.745,0.806,0.774


## Step 5 — Match criteria by position (Tsai et al., 2006)

The relaxed criteria follow Tsai et al. (2006), *Various criteria in the evaluation of
biomedical named entity recognition*, BMC Bioinformatics 7:92. Like the paper, they compare
where two annotations sit in the text: each term is located in the claim text, and two terms
match only if their positions fit the criterion.

| Criterion | Tsai et al. definition | Here: position in the claim text |
|---|---|---|
| **Exact** | Both boundaries (and the class) coincide | Same term, ignoring case, hyphen vs space and trailing punctuation (Step 2). Such terms are found at the same places, so this is also a position match. |
| **Right** | The right boundary matches exactly | The two terms end at the same place |
| **Left** | The left boundary matches exactly | The two terms start at the same place |
| **Left/right** | Either boundary matches exactly | Right or left |
| **Approximate** | One NE is a substring of the other | One term lies completely inside the other |
| **Core-term** | The tagged NE contains the NE's core term, the part that identifies it. Core terms often have capital letters, digits or special symbols (`SAP` in `p54 SAP kinase`) and can only be identified by hand. | The terms overlap and share a core word: one with a capital letter, a digit, or a symbol other than a hyphen. An automatic stand-in for hand identification. |
| **Partial** | Any fragment of the NE is detected | The terms overlap |

Right, left, left/right and approximate are nested: every right or left match is also an
approximate match. Every core-term match is also a partial match. Exact matches are always
counted under every criterion.

**Criteria not reproduced.** *Multiple-tagging match* (BioCreAtIvE) needs a gold standard with
alternative acceptable annotations, which we do not have. *Class merging* needs entity
classes, which our terms do not have. *Fragment match* scores every token of the text
separately; since a term that occurs several times cannot be tied to one occurrence (see
below), the token counts would depend on a guess, so it is left out.

**Departures from Tsai et al.**

- **Positions are inferred.** The annotators listed terms, not their positions. Each term is
  located in the claim text: case-insensitive, hyphen or space between words, whole words only
  (`heat` is not found inside `heater`), after removing abbreviations as in Step 1. A term
  that occurs several times matches if any of its occurrences fits. A term that cannot be
  found can still match exactly, but cannot get a relaxed match; Step 6 lists these terms.
- **One-to-one matching.** The paper does not say whether one tagged NE can match several
  gold NEs. Here each term is used at most once: exact matches are made first, then the
  largest possible set of relaxed matches among the remaining terms (maximum
  matching). This keeps Tsai et al.'s ordering of the criteria from strictest to loosest,
  and the result does not depend on the order in which annotators listed their terms. One
  match count serves both precision and recall, which keeps F1 symmetric between annotators.
- **No gold standard.** Tsai scores systems against a gold standard. Here annotators are
  compared in pairs, and precision and recall are reported with each annotator as the reference.

In [8]:
CLAIM_COL = "claim_text"


def raw_words(term):
    """Whitespace-separated words, edge punctuation removed, case kept: 'p54 SAP kinase.' -> ['p54', 'SAP', 'kinase']."""
    tokens = (token.strip(".,;:") for token in term.split())
    return [token for token in tokens if token]


CORE_WORD_RE = re.compile(r"[A-Z0-9]|[^A-Za-z0-9\-]")


def core_words(term):
    """Words with a capital letter, a digit or a symbol other than a hyphen: 'p54 SAP kinase' -> {'p54', 'sap'}."""
    return {token.lower() for token in raw_words(term) if CORE_WORD_RE.search(token)}


def shares_core_word(term_a, term_b):
    return bool(core_words(term_a) & core_words(term_b))


def term_pattern(term):
    """Regex for a term in the claim text: case-insensitive, hyphen or space between words, whole words only."""
    parts = [re.escape(p) for p in re.split(r"[\s\-]+", term.strip().rstrip(".,;:")) if p]
    return re.compile(r"(?<!\w)" + r"[\s\-]+".join(parts) + r"(?!\w)", re.IGNORECASE)


def find_spans(term, claim_text):
    """All (start, end) character positions of a term in the claim text."""
    return [m.span() for m in term_pattern(term).finditer(claim_text)]


def overlap(a, b):
    return a[0] < b[1] and b[0] < a[1]


def nested(a, b):
    """One span lies completely inside the other."""
    return (a[0] <= b[0] and b[1] <= a[1]) or (b[0] <= a[0] and a[1] <= b[1])


# Span relations, in the order the criteria are reported.
SPAN_RELATIONS = {
    "right": lambda a, b: a[1] == b[1],
    "left": lambda a, b: a[0] == b[0],
    "left_right": lambda a, b: a[0] == b[0] or a[1] == b[1],
    "approximate": nested,
    "core_term": overlap,  # plus a shared core word, see positional_criteria
    "partial": overlap,
}
CRITERION_ORDER = ["exact"] + list(SPAN_RELATIONS)


def positional_criteria(spans):
    """Criteria for one claim; spans maps each term to its (start, end) positions. None = exact matches only."""
    def make(relation, needs_core_word):
        def criterion(term_a, term_b):
            if needs_core_word and not shares_core_word(term_a, term_b):
                return False
            return any(relation(sa, sb) for sa in spans[term_a] for sb in spans[term_b])
        return criterion
    criteria = {"exact": None}
    for name, relation in SPAN_RELATIONS.items():
        criteria[name] = make(relation, needs_core_word=(name == "core_term"))
    return criteria


def count_matches(list_a, list_b, criterion):
    """Exact matches, then the largest possible set of one-to-one relaxed matches among the leftovers."""
    exact, leftover_a, leftover_b = find_exact_matches(list_a, list_b)
    if criterion is None or not leftover_a or not leftover_b:
        return len(exact), []
    m = np.array([[criterion(a, b) for b in leftover_b] for a in leftover_a], dtype=int)
    rows, cols = linear_sum_assignment(m, maximize=True)
    relaxed = [(leftover_a[i], leftover_b[j]) for i, j in zip(rows, cols) if m[i, j]]
    return len(exact) + len(relaxed), relaxed


APPROXIMATE_TYPES = ["same_span", "right", "left", "middle"]


def approximate_match_type(spans_a, spans_b):
    """Label an approximate match for review: 'same_span', 'right', 'left' or 'middle'."""
    def label(a, b):
        if a == b:
            return "same_span"
        if a[1] == b[1]:
            return "right"
        if a[0] == b[0]:
            return "left"
        return "middle"
    labels = {label(a, b) for a in spans_a for b in spans_b if nested(a, b)}
    return min(labels, key=APPROXIMATE_TYPES.index)


demo_claim = "wherein the plate heat exchanger transfers heat to a heat sink"
demo_terms = ["heat", "heat exchanger", "plate heat", "plate heat exchanger", "exchanger transfers", "heat sink"]
demo = positional_criteria({t: find_spans(t, demo_claim) for t in demo_terms})
print(find_spans("heat", demo_claim))                                 # three occurrences
print(demo["right"]("heat exchanger", "plate heat exchanger"))       # True: same end
print(demo["left"]("plate heat", "plate heat exchanger"))            # True: same start
print(demo["approximate"]("heat", "plate heat exchanger"))           # True: inside
print(demo["partial"]("plate heat exchanger", "exchanger transfers"))  # True: they overlap
print(demo["partial"]("heat exchanger", "heat sink"))                # False: shared word, different place
print(core_words("p54 SAP kinase"))                                  # {'p54', 'sap'}

[(18, 22), (43, 47), (53, 57)]
True
True
True
True
False
{'sap', 'p54'}


## Step 6 — Locate terms and count matches

For every claim, each annotator's terms are located in the claim text (Step 5), and each
pair of annotators is scored under each criterion. `df_not_found` lists the terms that
cannot be found in the claim text: they can still match exactly but get no relaxed match,
so if there are many, the relaxed scores are too low. Typical causes are typos, reworded
terms, and special characters written differently from the claim text.

`df_approximate_review` lists every approximate match, labelled by where the two terms sit:
`same_span` (the same place), `right`, `left` or `middle` (middle is
our own label, not a Tsai criterion), so they can be checked by eye.

In [9]:
# Terms are searched in the claim text after removing abbreviations, as in Step 1.
criteria_rows, review_rows, not_found_rows = [], [], []
for idx, row in df_raw.iterrows():
    claim_text = remove_abbreviations(str(row[CLAIM_COL]))
    terms = {ann: normalize_terms(row[ann]) for ann in ANNOTATORS}

    spans = {}
    for ann, term_list in terms.items():
        for term in term_list:
            spans[term] = find_spans(term, claim_text)
            if not spans[term]:
                not_found_rows.append({"patent_id": row["patent_id"], "claim_number": row["claim_number"],
                                       "annotator": ann, "term": term})
    criteria = positional_criteria(spans)

    for ann_a, ann_b in PAIRS:
        list_a, list_b = terms[ann_a], terms[ann_b]
        claim = {"patent_id": row["patent_id"], "claim_number": row["claim_number"],
                 "annotator_a": ann_a, "annotator_b": ann_b}
        for name, criterion in criteria.items():
            n_matches, relaxed = count_matches(list_a, list_b, criterion)
            criteria_rows.append({**claim, "criterion": name, "n_matches": n_matches,
                                  "n_a": len(list_a), "n_b": len(list_b)})
            if name == "approximate":
                for a_term, b_term in relaxed:
                    review_rows.append({
                        **claim, "term_a": a_term, "term_b": b_term,
                        "match_type": approximate_match_type(spans[a_term], spans[b_term]),
                        "single_word": min(len(raw_words(a_term)), len(raw_words(b_term))) == 1,
                    })

df_criteria_detail = pd.DataFrame(criteria_rows)
df_approximate_review = pd.DataFrame(review_rows)
df_not_found = pd.DataFrame(not_found_rows)

n_terms_total = sum(len(normalize_terms(row[ann])) for _, row in df_raw.iterrows() for ann in ANNOTATORS)
print(f"Terms not found in the claim text: {len(df_not_found)} of {n_terms_total}")
if len(df_not_found):
    print(df_not_found["annotator"].value_counts().to_string())
df_not_found

Terms not found in the claim text: 33 of 1906
annotator
Mark      16
Gloria    15
Tianyu     2


,patent_id,claim_number,annotator,term
0,7767672,54,Gloria,halogen F
1,7767672,54,Gloria,halogen J
2,7915258,8,Mark,"benzyl,[3,5-difluoro-4-(trifluoromethyl)phenyl]methyl"
3,7915258,8,Mark,R 2 is 4-bromophenyl
4,7915258,8,Mark,4-[(2-acetamidoethyl)thio]phenyl or 4-[[3-[(methylsulfonyl)amino]propyl]thio]phenyl
5,7915258,8,Mark,R 3 is 2-chlorophenyl
6,7915258,8,Mark,2-chloro-4-[[2-[(trifluoroacetyl)amino]ethyl]thio]phenyl or 2-chloro-4-[[2-[(cyclopropylcarbonyl)amino]ethyl]thio]phenyl
7,7915258,8,Mark,and R 4 is hydrogen atom
8,7915258,8,Mark,methoxy or hydroxyl
9,7951788,2,Gloria,C2-C18 acyl group


## Step 7 — Pooled Precision / Recall / F1 for every criterion

As in Tsai et al., each criterion is scored **independently** on corpus-level counts:
matches and terms are summed over all 100 claims, then

- `Precision = matches / terms of the other annotator`
- `Recall = matches / terms of the reference annotator`
- `F1 = 2PR / (P + R)`

Each pair is reported in both reference directions; F1 is the same in both. `f1_per_claim`
averages F1 per claim in the same way as Steps 3-4, kept as a supplementary number.

**Which numbers to report.** Tsai et al. found right match to be the relaxed criterion
closest to evaluation against multiple acceptable annotations (r = 0.979), with left match
second by a slight margin (r = 0.969), and suggest using both. Report exact F1 as the
headline, right-match F1 as the relaxed figure, and left-match F1 as a check on it.

**Statistics not reproduced.** Tsai et al. compared each criterion's F-scores with
BioCreAtIvE's multiple-tagging F-scores using Pearson correlation and a t-test over
20 system/dataset samples. That needs a multiple-tagging reference and many systems; here
there is no reference and only three annotator pairs, so it cannot be repeated.

In [10]:
summary_rows = []
for (name, ann_a, ann_b), grp in df_criteria_detail.groupby(["criterion", "annotator_a", "annotator_b"], sort=False):
    n_matches, n_a, n_b = grp["n_matches"].sum(), grp["n_a"].sum(), grp["n_b"].sum()
    f1_per_claim = grp.apply(lambda r: prf(r["n_matches"], r["n_a"], r["n_b"])[2], axis=1).mean()
    for reference, n_ref, n_other in [(ann_a, n_a, n_b), (ann_b, n_b, n_a)]:
        precision, recall = safe_div(n_matches, n_other), safe_div(n_matches, n_ref)
        summary_rows.append({
            "criterion": name, "annotator_a": ann_a, "annotator_b": ann_b, "reference": reference,
            "n_matches": n_matches, "n_ref": n_ref, "n_other": n_other,
            "precision": precision, "recall": recall, "f1": f1_from_pr(precision, recall),
            "f1_per_claim": f1_per_claim,
        })
df_summary_criteria = pd.DataFrame(summary_rows)

# Sanity check: exact-match scores must equal the strict pooled scores from Step 4.
exact_f1 = df_summary_criteria.query("criterion == 'exact'").set_index(["annotator_a", "annotator_b", "reference"])["f1"]
strict_f1 = df_summary_pooled.set_index(["annotator_a", "annotator_b", "reference"])["f1_strict_pooled"]
assert (exact_f1.sort_index() - strict_f1.sort_index()).abs().max() < 1e-12

# Sanity check: looser criteria never have fewer matches than stricter ones (Tsai et al.'s ordering).
n = (df_summary_criteria.drop_duplicates(["criterion", "annotator_a", "annotator_b"])
     .set_index(["criterion", "annotator_a", "annotator_b"])["n_matches"])
for looser, stricter in [("right", "exact"), ("left", "exact"), ("left_right", "right"),
                         ("left_right", "left"), ("approximate", "left_right"),
                         ("core_term", "exact"), ("partial", "approximate"), ("partial", "core_term")]:
    assert (n.loc[looser].values >= n.loc[stricter].values).all(), (looser, stricter)

print(f"Approximate matches: {len(df_approximate_review)}")
print(df_approximate_review["match_type"].value_counts().to_string())

# Pooled F1 per criterion (rows) and pair (columns).
display(
    df_summary_criteria.drop_duplicates(["criterion", "annotator_a", "annotator_b"])
    .assign(pair=lambda d: d["annotator_a"] + "-" + d["annotator_b"])
    .pivot(index="criterion", columns="pair", values="f1")
    .loc[CRITERION_ORDER].round(3)
)

# Full table: pooled P / R / F1 with the first annotator of each pair as the reference.
(df_summary_criteria.query("reference == annotator_a")
 .set_index(["criterion", "annotator_a", "annotator_b"])
 [["n_matches", "n_ref", "n_other", "precision", "recall", "f1", "f1_per_claim"]]
 .loc[CRITERION_ORDER].round(3))

Approximate matches: 149
match_type
right     79
left      61
middle     9


pair,Mark-Gloria,Tianyu-Gloria,Tianyu-Mark
criterion,,,
exact,0.644,0.668,0.774
right,0.696,0.734,0.833
left,0.677,0.702,0.814
left_right,0.708,0.754,0.855
approximate,0.709,0.756,0.855
core_term,0.657,0.683,0.784
partial,0.709,0.756,0.855


n_matches  n_ref  n_other  precision  \
criterion   annotator_a annotator_b                                         
exact       Tianyu      Mark               543    674      729      0.745   
                        Gloria             393    674      503      0.781   
            Mark        Gloria             397    729      503      0.789   
right       Tianyu      Mark               584    674      729      0.801   
                        Gloria             432    674      503      0.859   
            Mark        Gloria             429    729      503      0.853   
left        Tianyu      Mark               571    674      729      0.783   
                        Gloria             413    674      503      0.821   
            Mark        Gloria             417    729      503      0.829   
left_right  Tianyu      Mark               600    674      729      0.823   
                        Gloria             444    674      503      0.883   
            Mark        Gloria             436    729      503      0.867   
approximate Tianyu      Mark               600    674      729      0.823   
                        Gloria             445    674      503      0.885   
            Mark        Gloria             437    729      503      0.869   
core_term   Tianyu      Mark               550    674      729      0.754   
                        Gloria             402    674      503      0.799   
            Mark        Gloria             405    729      503      0.805   
partial     Tianyu      Mark               600    674      729      0.823   
                        Gloria             445    674      503      0.885   
            Mark        Gloria             437    729      503      0.869   

                                     recall     f1  f1_per_claim  
criterion   annotator_a annotator_b                               
exact       Tianyu      Mark          0.806  0.774         0.728  
                        Gloria        0.583  0.668         0.686  
            Mark        Gloria        0.545  0.644         0.644  
right       Tianyu      Mark          0.866  0.833         0.796  
                        Gloria        0.641  0.734         0.777  
            Mark        Gloria        0.588  0.696         0.710  
left        Tianyu      Mark          0.847  0.814         0.781  
                        Gloria        0.613  0.702         0.731  
            Mark        Gloria        0.572  0.677         0.691  
left_right  Tianyu      Mark          0.890  0.855         0.826  
                        Gloria        0.659  0.754         0.801  
            Mark        Gloria        0.598  0.708         0.728  
approximate Tianyu      Mark          0.890  0.855         0.826  
                        Gloria        0.660  0.756         0.803  
            Mark        Gloria        0.599  0.709         0.728  
core_term   Tianyu      Mark          0.816  0.784         0.740  
                        Gloria        0.596  0.683         0.707  
            Mark        Gloria        0.556  0.657         0.660  
partial     Tianyu      Mark          0.890  0.855         0.826  
                        Gloria        0.660  0.756         0.803  
            Mark        Gloria        0.599  0.709         0.728

In [11]:
# Approximate matches for review: middle and left first, single-word matches first within each type.
df_approximate_review.sort_values(
    ["match_type", "single_word"],
    key=lambda col: col.map(APPROXIMATE_TYPES.index) if col.name == "match_type" else ~col,
    kind="stable",
    ascending=[False, True],
)

,patent_id,claim_number,annotator_a,annotator_b,term_a,term_b,match_type,single_word
8,7915258,8,Mark,Gloria,4-(methoxy-carbonyl)phenyl,methoxy,middle,True
83,9561309,20,Tianyu,Mark,copolymers,polymers and copolymers of PEG acrylate,middle,True
30,8668684,19,Tianyu,Mark,hydraulically or pneumatically spreadable expansion element,pneumatically spreadable,middle,False
59,9023024,1,Tianyu,Gloria,delivery device,microwave energy delivery device temperature,middle,False
85,9561309,20,Tianyu,Mark,carboxylic acid,polymers and copolymers of carboxylic acid bearing monomers,middle,False
...,...,...,...,...,...,...,...,...
140,10459249,12,Mark,Gloria,optical powers,front surface optical powers,right,False
145,10478795,11,Tianyu,Gloria,shell component,second shell component,right,False
146,10478795,11,Mark,Gloria,shell component,second shell component,right,False
147,10494432,71,Tianyu,Gloria,B-cell lymphoma,diffuse large B-cell lymphoma,right,False


## Results

Under exact matching, pooled F1 was 0.774 (Tianyu–Mark), 0.668 (Tianyu–Gloria) and 0.644
(Mark–Gloria). Under right match, the relaxed criterion Tsai et al. (2006) found closest to
evaluation against multiple acceptable annotations, F1 rose to 0.833, 0.734 and 0.696. Left
match gave slightly lower values (0.814, 0.702, 0.677), the same order as in the paper.

Much of the disagreement comes from how many terms were extracted rather than from term
boundaries. Gloria extracted 503 terms, against 674 (Tianyu) and 729 (Mark). With Tianyu or
Mark as the reference, pairs with Gloria therefore show high precision (0.85–0.86 under right
match) but low recall (0.59–0.64).

Partial match gave the same scores as approximate match: allowing terms that only overlap,
without one containing the other, added no matches.

33 of 1,906 terms (1.7%) could not be located in the claim text, mainly because of typos,
phrases that combine several terms, rewording, and special characters. They can match exactly
but receive no relaxed match.

*These figures come from the current run (Step 7); update them if the data or the rules change.*